# Chapter 10 — Ablation: where does the GraphGPS gain on ZINC come from?

*Hands-On Graph Neural Networks Using Python*, Second Edition.

The chapter compares a GINE baseline against GraphGPS. Between those two models, three things change at once: the **global attention**, the **positional encodings**, and the rest of the **GraphGPS block** (residual connections, a feed-forward layer, dropout, and two more layers). This notebook separates them, over three seeds, with six variants:

| variant | block | attention | PE |
|---|---|---|---|
| GINE baseline | the chapter's 4-layer, 64-hidden GINE | no | no |
| local only, no PE | GraphGPS block with the attention branch removed | no | no |
| local only, with PE | same | no | yes |
| local only, widened | same, wider, matched in parameters to the full model | no | no |
| attention, no PE | full GraphGPS block | yes | no |
| attention, with PE | full GraphGPS block (the chapter's GraphGPS) | yes | yes |

The "local only" variants keep everything in the GraphGPS block except the attention branch: the GINE convolution with its residual connection and normalization, and the feed-forward layer with dropout. The only difference from the full model is attention.

**Runtime:** set the Colab runtime to a GPU (Runtime > Change runtime type > A100). On an A100 the full grid of 6 variants x 3 seeds takes about two hours.

## 1. Install

In [1]:
!pip -q install torch-geometric
import torch, torch_geometric
print('torch', torch.__version__, '| PyG', torch_geometric.__version__)
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 4.8 MB/s eta 0:00:00
torch 2.11.0+cu130 | PyG 2.8.1
CUDA device: NVIDIA A100-SXM4-40GB


## 2. Configuration

`SEEDS` and `EPOCHS` are the two knobs that control runtime. Drop to `SEEDS = [0]` and `EPOCHS = 30` for a quick smoke test before committing to the full grid.

In [2]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.nn import (BatchNorm1d, Dropout, Embedding, Linear, ModuleList,
                      ReLU, Sequential)

from torch_geometric.datasets import ZINC
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, GPSConv, global_add_pool
from torch_geometric.nn.resolver import normalization_resolver
from torch_geometric.transforms import AddLaplacianEigenvectorPE

SEEDS      = [0, 1, 2]
EPOCHS     = 150
HIDDEN     = 96
NUM_LAYERS = 6
PE_DIM     = 8
HEADS      = 8
DROPOUT    = 0.1
BATCH_SIZE = 128
OUT_CSV    = Path('ablation_results.csv')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type != 'cuda':
    print('WARNING: no GPU found. On CPU this grid takes many hours.')

Using device: cuda


## 3. Data

One copy of ZINC-subset with Laplacian positional encodings, shared by every variant. The variants without PE simply ignore the `pe` attribute. The pre-transform runs once and is cached, so this cell is slow only the first time.

In [3]:
pe_transform = AddLaplacianEigenvectorPE(k=PE_DIM, attr_name='pe',
                                        is_undirected=True)
splits = {}
for split in ('train', 'val', 'test'):
    splits[split] = ZINC(root='./data/ZINC_PE', subset=True, split=split,
                         pre_transform=pe_transform)
print('ZINC-subset: ' + ', '.join(f'{k}={len(v)}' for k, v in splits.items()))
print('sample:', splits['train'][0])

Extracting data/ZINC_PE/molecules.zip
Processing...
Processing test dataset: 100%|██████████| 1000/1000 [00:01<00:00, 987.43it/s]

ZINC-subset: train=10000, val=1000, test=1000
sample: Data(x=[29, 1], edge_index=[2, 64], edge_attr=[64], y=[1], pe=[29, 8])



Done!


## 4. The models

`GINEBaseline` is the model of the chapter's second section, unchanged. `LocalGPSBlock` is PyG's `GPSConv` (version 2.8) with the attention branch removed: the local convolution with dropout, residual connection and normalization, followed by the same feed-forward layer and final normalization. `Variant` stacks either `LocalGPSBlock` or `GPSConv` layers, with or without positional encodings. The sign flip on the positional encodings is sampled per graph and per eigenvector, as in the chapter.

In [4]:
class GINEBaseline(torch.nn.Module):
    def __init__(self, hidden_channels=64, num_layers=4,
                 num_atom_types=28, num_bond_types=4):
        super().__init__()
        self.atom_embedding = Embedding(num_atom_types, hidden_channels)
        self.bond_embedding = Embedding(num_bond_types, hidden_channels)

        self.convs = ModuleList()
        self.bns   = ModuleList()
        for _ in range(num_layers):
            nn_mlp = Sequential(
                Linear(hidden_channels, hidden_channels),
                ReLU(),
                Linear(hidden_channels, hidden_channels),
            )
            self.convs.append(GINEConv(nn_mlp, train_eps=True))
            self.bns.append(BatchNorm1d(hidden_channels))

        self.head = Sequential(
            Linear(hidden_channels, hidden_channels),
            ReLU(),
            Linear(hidden_channels, 1),
        )

    def forward(self, x, pe, edge_index, edge_attr, batch):
        h = self.atom_embedding(x.squeeze(-1))
        e = self.bond_embedding(edge_attr)
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, edge_index, e)
            h = bn(h)
            h = F.relu(h)
        h = global_add_pool(h, batch)
        return self.head(h).squeeze(-1)


class LocalGPSBlock(torch.nn.Module):
    """GPSConv without its attention branch."""
    def __init__(self, channels, conv, dropout=0.0, norm='batch_norm'):
        super().__init__()
        self.conv = conv
        self.dropout = dropout
        self.mlp = Sequential(
            Linear(channels, channels * 2),
            ReLU(),
            Dropout(dropout),
            Linear(channels * 2, channels),
            Dropout(dropout),
        )
        self.norm1 = normalization_resolver(norm, channels)
        self.norm3 = normalization_resolver(norm, channels)

    def forward(self, x, edge_index, batch=None, **kwargs):
        h = self.conv(x, edge_index, **kwargs)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.norm1(h + x)
        out = h + self.mlp(h)
        return self.norm3(out)


class Variant(torch.nn.Module):
    def __init__(self, use_attn, use_pe,
                 hidden_channels=HIDDEN, num_layers=NUM_LAYERS,
                 pe_dim=PE_DIM, heads=HEADS, dropout=DROPOUT,
                 num_atom_types=28, num_bond_types=4):
        super().__init__()
        self.use_pe = use_pe
        self.atom_embedding = Embedding(num_atom_types, hidden_channels)
        self.bond_embedding = Embedding(num_bond_types, hidden_channels)
        self.pe_lin = Linear(pe_dim, hidden_channels) if use_pe else None

        self.convs = ModuleList()
        for _ in range(num_layers):
            local_nn = Sequential(
                Linear(hidden_channels, hidden_channels),
                ReLU(),
                Linear(hidden_channels, hidden_channels),
            )
            gine = GINEConv(local_nn, train_eps=True)
            if use_attn:
                self.convs.append(GPSConv(hidden_channels, conv=gine,
                                          heads=heads, dropout=dropout,
                                          attn_type='multihead'))
            else:
                self.convs.append(LocalGPSBlock(hidden_channels, conv=gine,
                                                dropout=dropout))

        self.head = Sequential(
            Linear(hidden_channels, hidden_channels),
            ReLU(),
            Linear(hidden_channels, 1),
        )

    def forward(self, x, pe, edge_index, edge_attr, batch):
        h = self.atom_embedding(x.squeeze(-1))
        if self.use_pe:
            if self.training:
                num_graphs = int(batch.max().item()) + 1
                sign = torch.randint(0, 2, (num_graphs, pe.size(1)),
                                     device=pe.device) * 2 - 1
                pe = pe * sign[batch]
            h = h + self.pe_lin(pe)
        e = self.bond_embedding(edge_attr)
        for conv in self.convs:
            h = conv(h, edge_index, batch, edge_attr=e)
        h = global_add_pool(h, batch)
        return self.head(h).squeeze(-1)


def n_params(model):
    return sum(p.numel() for p in model.parameters())


full_params = n_params(Variant(use_attn=True, use_pe=True))
HIDDEN_WIDE = min(range(96, 257),
                  key=lambda h: abs(n_params(Variant(use_attn=False, use_pe=False,
                                                     hidden_channels=h))
                                    - full_params))
print(f'full GraphGPS: {full_params:,} parameters')
print(f'local only, widened: hidden={HIDDEN_WIDE}, '
      f'{n_params(Variant(use_attn=False, use_pe=False, hidden_channels=HIDDEN_WIDE)):,} parameters')

full GraphGPS: 574,951 parameters
local only, widened: hidden=123, 570,604 parameters


## 5. Train and evaluate one variant

In [5]:
criterion = torch.nn.L1Loss()


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_error = 0.0
    total_n = 0
    for batch in loader:
        batch = batch.to(device)
        pred = model(batch.x, batch.pe, batch.edge_index,
                     batch.edge_attr, batch.batch)
        total_error += (pred - batch.y).abs().sum().item()
        total_n += batch.y.size(0)
    return total_error / total_n


def run_one(make_model, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)

    train_loader = DataLoader(splits['train'], batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(splits['val'],   batch_size=BATCH_SIZE)
    test_loader  = DataLoader(splits['test'],  batch_size=BATCH_SIZE)

    model = make_model().to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)

    best_val, best_state = float('inf'), None
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optim.zero_grad()
            pred = model(batch.x, batch.pe, batch.edge_index,
                         batch.edge_attr, batch.batch)
            loss = criterion(pred, batch.y)
            loss.backward(); optim.step()
        sched.step()
        val_mae = evaluate(model, val_loader)
        if val_mae < best_val:
            best_val = val_mae
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    seconds = time.time() - t0

    model.load_state_dict(best_state)
    return {'seed': seed, 'params': n_params(model), 'val_mae': best_val,
            'test_mae': evaluate(model, test_loader),
            'sec_per_epoch': seconds / EPOCHS}

## 6. Run the grid

Results are written to `ablation_results.csv` after every run, so a disconnected session keeps what it has already produced.

In [6]:
VARIANTS = [
    ('GINE baseline (4 layers, 64 hidden)', lambda: GINEBaseline()),
    ('local only, no PE',   lambda: Variant(use_attn=False, use_pe=False)),
    ('local only, with PE', lambda: Variant(use_attn=False, use_pe=True)),
    (f'local only, widened (hidden={HIDDEN_WIDE})',
     lambda: Variant(use_attn=False, use_pe=False, hidden_channels=HIDDEN_WIDE)),
    ('attention, no PE',    lambda: Variant(use_attn=True, use_pe=False)),
    ('attention, with PE (full GraphGPS)',
     lambda: Variant(use_attn=True, use_pe=True)),
]

records = []
for label, make_model in VARIANTS:
    for seed in SEEDS:
        print(f'\n--- {label} | seed {seed} ---')
        rec = run_one(make_model, seed)
        rec['label'] = label
        records.append(rec)
        print(f"    test MAE {rec['test_mae']:.4f} | params {rec['params']:,} | "
              f"{rec['sec_per_epoch']:.1f}s per epoch")
        pd.DataFrame(records).to_csv(OUT_CSV, index=False)


--- GINE baseline (4 layers, 64 hidden) | seed 0 ---
    test MAE 0.2332 | params 40,069 | 1.1s per epoch

--- GINE baseline (4 layers, 64 hidden) | seed 1 ---
    test MAE 0.2875 | params 40,069 | 1.1s per epoch

--- GINE baseline (4 layers, 64 hidden) | seed 2 ---
    test MAE 0.2447 | params 40,069 | 1.1s per epoch

--- local only, no PE | seed 0 ---
    test MAE 0.1989 | params 349,447 | 1.8s per epoch

--- local only, no PE | seed 1 ---
    test MAE 0.1838 | params 349,447 | 1.8s per epoch

--- local only, no PE | seed 2 ---
    test MAE 0.1934 | params 349,447 | 1.8s per epoch

--- local only, with PE | seed 0 ---
    test MAE 0.1885 | params 350,311 | 1.8s per epoch

--- local only, with PE | seed 1 ---
    test MAE 0.1925 | params 350,311 | 1.8s per epoch

--- local only, with PE | seed 2 ---
    test MAE 0.1886 | params 350,311 | 1.8s per epoch

--- local only, widened (hidden=123) | seed 0 ---
    test MAE 0.1784 | params 570,604 | 1.8s per epoch

--- local only, widened (hi

## 7. Summary table

In [7]:
df = pd.DataFrame(records)
summary = (df.groupby('label', sort=False)
             .agg(params=('params', 'first'),
                  mae_mean=('test_mae', 'mean'),
                  mae_std=('test_mae', 'std'),
                  sec_epoch=('sec_per_epoch', 'mean')))

print('=' * 78)
print(f'Ablation on ZINC-subset | {len(SEEDS)} seeds | {EPOCHS} epochs | '
      f'{NUM_LAYERS} layers | {HIDDEN} hidden')
print('=' * 78)
for label, row in summary.iterrows():
    print(f'{label:<38} {row.mae_mean:.4f} +/- {row.mae_std:.4f} | '
          f'{int(row.params):>8,} params | {row.sec_epoch:5.1f}s/epoch')
print('=' * 78)
summary

Ablation on ZINC-subset | 3 seeds | 150 epochs | 6 layers | 96 hidden
GINE baseline (4 layers, 64 hidden)    0.2551 +/- 0.0286 |   40,069 params |   1.1s/epoch
local only, no PE                      0.1920 +/- 0.0077 |  349,447 params |   1.8s/epoch
local only, with PE                    0.1898 +/- 0.0023 |  350,311 params |   1.8s/epoch
local only, widened (hidden=123)       0.1795 +/- 0.0011 |  570,604 params |   1.8s/epoch
attention, no PE                       0.2210 +/- 0.0062 |  574,087 params |   3.3s/epoch
attention, with PE (full GraphGPS)     0.1941 +/- 0.0021 |  574,951 params |   3.4s/epoch


,params,mae_mean,mae_std,sec_epoch
label,,,,
"GINE baseline (4 layers, 64 hidden)",40069,0.255142,0.028576,1.124085
"local only, no PE",349447,0.192043,0.007687,1.780506
"local only, with PE",350311,0.189847,0.002305,1.811460
"local only, widened (hidden=123)",570604,0.179457,0.001059,1.777646
"attention, no PE",574087,0.220956,0.006171,3.328582
"attention, with PE (full GraphGPS)",574951,0.194144,0.002054,3.375529


## 8. Download the results

Run this to pull `ablation_results.csv` down from the Colab session.

In [8]:
from google.colab import files
files.download(str(OUT_CSV))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>